# Aether Vision — YOLO training with QAT for the QCS6490

Trains a YOLO detector on a dataset made with `training/annotate.py pack`, runs
**quantization-aware training (QAT)** and exports an **int8 TFLite** model for the
**QCS6490 NPU** (Hexagon HTP, through the QNN TFLite delegate that `aether_vision.detector` loads).

This notebook is self-contained: it does not import the aether-vision project, so it runs on Kaggle
(or any machine with a GPU).

| Step | Output |
|---|---|
| 1. Float training (Ultralytics fine-tune) | `runs/float/weights/best.pt` |
| 2. QAT (PyTorch PT2E fake-quant int8, YOLO loss) | `runs/qat/qat_state.pt` |
| 3. Export: int8 TFLite, NHWC input, one `[1, 4+nc, N]` output, normalized boxes | `model_int8.tflite`, `labels.txt` |
| 4. Verify: mAP of float vs PTQ vs QAT vs TFLite; optional AI Hub profile on a QCS6490 | `results.json` |

**On Kaggle**

1. Upload `training/dataset.zip` as a dataset (*Datasets → New dataset*); Kaggle unpacks it.
2. In this notebook: *Add input* → your dataset; *Settings* → **Accelerator: GPU T4** (the P100 is too old
   for current PyTorch) and **Internet on** (for pip and the pretrained weights).
3. *Run all*. Download `aether_model.zip` from the output panel (`/kaggle/working`).

On the robot, put the `.tflite` and `labels.txt` in `models/` and set `detector.model_path` and
`detector.labels_path` in the dashboard.

In [ ]:
# Ultralytics is pinned below 8.4 because step 2 relies on the YOLO11 Detect head.
# ai-edge-torch pins its own torch version; installing it first lets pip pick a matching one.
%pip install -q "ai-edge-torch>=0.5" "ultralytics>=8.3.40,<8.4"

## 0. Setup

In [ ]:
import copy
import json
import os
import random
import shutil
import time
import zipfile
from pathlib import Path

os.environ.setdefault(
    "WANDB_MODE", "disabled"
)  # Kaggle ships wandb; Ultralytics would prompt for a key

import cv2
import numpy as np
import torch
import yaml
from IPython.display import Image, display
from ultralytics import YOLO, settings

settings.update({"wandb": False})

ON_KAGGLE = Path("/kaggle/input").exists()
WORK = Path("/kaggle/working") if ON_KAGGLE else Path.cwd() / "work"
WORK.mkdir(parents=True, exist_ok=True)

# ---- Settings --------------------------------------------------------------------------------
IMGSZ = 640  # square model input; the 640x480 camera image is letterboxed into it
BASE_WEIGHTS = "yolo11n.pt"  # n is fastest on the HTP; yolo11s.pt if accuracy is short
FLOAT_EPOCHS = 100
BATCH = 16
QAT_EPOCHS = 15
QAT_FREEZE_EPOCHS = 3  # final QAT epochs with observers frozen and BatchNorm in eval mode
QAT_LR = 2e-5
SEED = 0
# Outside Kaggle: a dataset.zip from `annotate.py pack`, or a folder containing data.yaml.
LOCAL_DATASET = Path("dataset.zip")
# -----------------------------------------------------------------------------------------------


def find_dataset() -> Path:
    """Copy the dataset to a writable folder (Ultralytics writes label caches next to labels)."""
    target = WORK / "dataset"
    if (target / "data.yaml").exists():
        return target
    if ON_KAGGLE:
        found = sorted(Path("/kaggle/input").rglob("data.yaml"))
        if not found:
            raise FileNotFoundError("No data.yaml under /kaggle/input: add the dataset as an input")
        shutil.copytree(found[0].parent, target)
    elif LOCAL_DATASET.suffix == ".zip":
        with zipfile.ZipFile(LOCAL_DATASET) as archive:
            archive.extractall(WORK)  # the zip contains dataset/...
    else:
        shutil.copytree(LOCAL_DATASET, target)
    return target


DATASET = find_dataset()
DATA_YAML = DATASET / "data.yaml"
spec = yaml.safe_load(DATA_YAML.read_text())
spec["path"] = str(DATASET)
DATA_YAML.write_text(yaml.safe_dump(spec, sort_keys=False))
NAMES = [spec["names"][i] for i in sorted(spec["names"])]
RUNS = WORK / "runs"
OUT = WORK / "export"
OUT.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
VAL_IMAGES = sorted(
    p
    for p in (DATASET / "images" / "val").iterdir()
    if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
)
TRAIN_IMAGES = sorted(
    p
    for p in (DATASET / "images" / "train").iterdir()
    if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
)
print(f"dataset {DATASET}: {len(TRAIN_IMAGES)} train, {len(VAL_IMAGES)} val, classes {NAMES}")
print("device:", torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "cpu (slow!)")

### Helpers

Preprocessing, decoding and NMS mirror `aether_vision/detector.py`, so the numbers here are what the
robot gets. mAP is COCO-style (101-point, IoU 0.50 and 0.50–0.95).

In [ ]:
EVAL_CONF, EVAL_IOU, EVAL_MAX_DET = 0.01, 0.45, 100
IOU_THRESHOLDS = np.linspace(0.5, 0.95, 10)


def letterbox(image: np.ndarray, size: int = IMGSZ):
    """Keep aspect, pad gray (114), BGR -> RGB. Returns (image, scale, pad_x, pad_y)."""
    h, w = image.shape[:2]
    scale = min(size / w, size / h)
    new_w, new_h = round(w * scale), round(h * scale)
    pad_x, pad_y = (size - new_w) // 2, (size - new_h) // 2
    resized = cv2.resize(image, (new_w, new_h)) if (new_w, new_h) != (w, h) else image
    padded = cv2.copyMakeBorder(
        resized,
        pad_y,
        size - new_h - pad_y,
        pad_x,
        size - new_w - pad_x,
        cv2.BORDER_CONSTANT,
        value=(114, 114, 114),
    )
    return cv2.cvtColor(padded, cv2.COLOR_BGR2RGB), scale, pad_x, pad_y


def postprocess(y: np.ndarray, scale: float, pad_x: int, pad_y: int):
    """[4+nc, N] normalized xywh + class scores -> (xyxy image pixels, scores, classes) after NMS."""
    rows = y.T
    classes = rows[:, 4:].argmax(1)
    scores = rows[np.arange(len(rows)), 4 + classes]
    keep = scores >= EVAL_CONF
    rows, scores, classes = rows[keep], scores[keep], classes[keep]
    xy, wh = rows[:, :2] * IMGSZ, rows[:, 2:4] * IMGSZ
    boxes = (np.hstack((xy - wh / 2, xy + wh / 2)) - [pad_x, pad_y, pad_x, pad_y]) / scale
    xywh = np.column_stack((boxes[:, :2], boxes[:, 2:] - boxes[:, :2]))
    idx = np.asarray(
        cv2.dnn.NMSBoxesBatched(
            xywh.tolist(),
            scores.tolist(),
            classes.tolist(),
            EVAL_CONF,
            EVAL_IOU,
            top_k=EVAL_MAX_DET,
        ),
        dtype=np.int64,
    ).reshape(-1)
    return boxes[idx], scores[idx], classes[idx]


def read_labels(image_path: Path, width: int, height: int):
    """YOLO label file -> (class ids [n], boxes xyxy in pixels [n, 4])."""
    path = DATASET / "labels" / image_path.parent.name / (image_path.stem + ".txt")
    if not path.exists() or not path.read_text().strip():
        return np.zeros(0, np.int64), np.zeros((0, 4), np.float32)
    rows = np.loadtxt(path, ndmin=2, dtype=np.float32)
    cx, cy, bw, bh = (
        rows[:, 1] * width,
        rows[:, 2] * height,
        rows[:, 3] * width,
        rows[:, 4] * height,
    )
    return rows[:, 0].astype(np.int64), np.stack(
        (cx - bw / 2, cy - bh / 2, cx + bw / 2, cy + bh / 2), 1
    )


def box_iou(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    lt, rb = np.maximum(a[:, None, :2], b[None, :, :2]), np.minimum(a[:, None, 2:], b[None, :, 2:])
    inter = np.clip(rb - lt, 0, None).prod(2)
    area_a = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1])
    area_b = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / (area_a[:, None] + area_b[None] - inter + 1e-9)


def evaluate(predict, images=None, batch: int = BATCH) -> dict:
    """predict(list of BGR images) -> list of (boxes xyxy, scores, classes)."""
    images = images or VAL_IMAGES
    records, n_gt = [], np.zeros(len(NAMES), np.int64)
    for start in range(0, len(images), batch):
        paths = images[start : start + batch]
        frames = [cv2.imread(str(p)) for p in paths]
        for path, frame, (boxes, scores, classes) in zip(paths, frames, predict(frames)):
            gt_cls, gt_box = read_labels(path, frame.shape[1], frame.shape[0])
            n_gt += np.bincount(gt_cls, minlength=len(NAMES))
            iou = (
                box_iou(boxes, gt_box)
                if len(boxes) and len(gt_box)
                else np.zeros((len(boxes), len(gt_box)))
            )
            iou[classes[:, None] != gt_cls[None, :]] = 0
            tp = np.zeros((len(boxes), len(IOU_THRESHOLDS)), bool)
            for t, threshold in enumerate(IOU_THRESHOLDS):
                used = np.zeros(len(gt_box), bool)
                for i in np.argsort(-scores):  # greedy, most confident first
                    if len(gt_box):
                        candidates = np.where(~used & (iou[i] >= threshold), iou[i], -1)
                        j = int(candidates.argmax())
                        if candidates[j] >= 0:
                            used[j], tp[i, t] = True, True
            records += [(int(c), float(s), tp[i]) for i, (c, s) in enumerate(zip(classes, scores))]
    ap = np.zeros((len(NAMES), len(IOU_THRESHOLDS)))
    for c in range(len(NAMES)):
        rec = sorted((r for r in records if r[0] == c), key=lambda r: -r[1])
        if not n_gt[c] or not rec:
            continue
        tps = np.cumsum(np.array([r[2] for r in rec]), axis=0)
        fps = np.arange(1, len(rec) + 1)[:, None] - tps
        recall, precision = tps / n_gt[c], tps / (tps + fps)
        for t in range(len(IOU_THRESHOLDS)):
            envelope = np.maximum.accumulate(precision[::-1, t])[::-1]
            idx = np.searchsorted(recall[:, t], np.linspace(0, 1, 101), side="left")
            ap[c, t] = np.where(idx < len(rec), envelope[np.minimum(idx, len(rec) - 1)], 0).mean()
    present = n_gt > 0
    return {"mAP50": float(ap[present, 0].mean()), "mAP50-95": float(ap[present].mean())}


def draw(image, boxes, scores, classes):
    out = image.copy()
    for (x1, y1, x2, y2), s, c in zip(boxes.astype(int), scores, classes):
        color = [(0, 200, 255), (255, 120, 0), (80, 220, 80), (220, 80, 220)][int(c) % 4]
        cv2.rectangle(out, (x1, y1), (x2, y2), color, 2)
        cv2.putText(out, f"{NAMES[int(c)]} {s:.2f}", (x1, max(12, y1 - 4)), 0, 0.5, color, 1)
    return out


def show_grid(images, cols: int = 4, cell: int = 320):
    cells = [cv2.resize(i, (cell, round(i.shape[0] * cell / i.shape[1]))) for i in images]
    h = max(c.shape[0] for c in cells)
    cells = [cv2.copyMakeBorder(c, 0, h - c.shape[0], 0, 0, cv2.BORDER_CONSTANT) for c in cells]
    cells += [np.zeros_like(cells[0])] * (-len(cells) % cols)
    grid = np.vstack([np.hstack(cells[r : r + cols]) for r in range(0, len(cells), cols)])
    display(Image(cv2.imencode(".jpg", grid)[1].tobytes()))

## 1. Float training

A normal Ultralytics fine-tune from COCO weights: the starting point for QAT and the accuracy the int8
model is compared against.

In [ ]:
float_model = YOLO(BASE_WEIGHTS)
float_model.train(
    data=str(DATA_YAML),
    imgsz=IMGSZ,
    epochs=FLOAT_EPOCHS,
    batch=BATCH,
    device=0 if DEVICE.type == "cuda" else "cpu",
    project=str(RUNS),
    name="float",
    exist_ok=True,
    seed=SEED,
    patience=30,
    close_mosaic=10,
    workers=4,
    plots=True,
)
FLOAT_WEIGHTS = RUNS / "float" / "weights" / "best.pt"
print("float weights:", FLOAT_WEIGHTS)

## 2. Quantization-aware training

Plain post-training quantization of YOLO to int8 often costs several mAP points, mostly in box
precision. QAT inserts int8 fake-quantization into the graph and keeps training, so the weights adapt to
the rounding.

1. The trained model is wrapped so its forward returns both the raw head feature maps (for the YOLO loss)
   and the **decoded** output `[B, 4+nc, N]` with boxes divided by the input size. Decoding (DFL softmax,
   anchors, sigmoid) is inside the quantized graph, so the NPU runs the whole model, and boxes and
   scores (both 0..1) share one int8 output scale without losing precision.
2. `torch.export` captures the graph and `prepare_qat_pt2e` inserts fake-quant with **ai-edge-torch's
   quantizer** (symmetric int8, per-channel weights, per-tensor activations — what the Hexagon HTP runs
   natively), so the TFLite uses exactly the scales learned here.
3. A short fine-tune with a low learning rate and no mosaic. In the last epochs the observers are frozen
   and BatchNorm uses running statistics, so the scales settle before export.

The graph is traced at a fixed batch size; the export re-traces at batch 1 and loads the same weights and
quantization parameters.

In [ ]:
import ai_edge_torch
from ai_edge_torch.quantize.pt2e_quantizer import PT2EQuantizer, get_symmetric_quantization_config
from ai_edge_torch.quantize.quant_config import QuantConfig
from torch.ao.quantization import (
    disable_observer,
    enable_observer,
    move_exported_model_to_eval,
    move_exported_model_to_train,
)
from torch.ao.quantization.quantize_pt2e import convert_pt2e, prepare_qat_pt2e
from ultralytics.cfg import get_cfg
from ultralytics.data import build_dataloader, build_yolo_dataset
from ultralytics.data.utils import check_det_dataset
from ultralytics.utils.loss import v8DetectionLoss

QAT_DIR = RUNS / "qat"
QAT_DIR.mkdir(parents=True, exist_ok=True)
_export = getattr(torch.export, "export_for_training", torch.export.export)
quantizer = PT2EQuantizer().set_global(
    get_symmetric_quantization_config(is_per_channel=True, is_qat=True)
)


class YoloQAT(torch.nn.Module):
    """Returns (decoded [B, 4+nc, N] with boxes normalized to 0..1, raw head feature maps)."""

    def __init__(self, detection_model: torch.nn.Module, imgsz: int) -> None:
        super().__init__()
        self.det = detection_model
        self.imgsz = float(imgsz)
        head = detection_model.model[-1]
        head.export, head.dynamic = False, False

    def forward(self, x):
        self.det.model[
            -1
        ].training = False  # the head decodes and returns (y, feats); the rest trains
        y, feats = self.det(x)
        return torch.cat((y[:, :4] / self.imgsz, y[:, 4:]), 1), feats


def load_float(weights: Path) -> torch.nn.Module:
    det = YOLO(str(weights)).model.float().cpu()
    for p in det.parameters():
        p.requires_grad_(True)
    return det


def make_qat_graph(det: torch.nn.Module, batch: int) -> torch.nn.Module:
    """Export the wrapped model at a fixed batch and insert fake-quant (on the CPU)."""
    wrapper = YoloQAT(copy.deepcopy(det), IMGSZ).train()
    example = (torch.rand(batch, 3, IMGSZ, IMGSZ),)
    graph = prepare_qat_pt2e(_export(wrapper, example).module(), quantizer)
    graph(*example)  # initialise the observers
    return graph


def torch_predictor(model, fixed_batch: int | None = None):
    """Predict function for a model returning (decoded, feats); pads the batch to fixed_batch."""

    def predict(images):
        prepared = [letterbox(i) for i in images]
        x = torch.from_numpy(np.stack([p[0] for p in prepared])).permute(0, 3, 1, 2).float() / 255
        if fixed_batch and len(x) < fixed_batch:
            x = torch.cat([x, x[:1].expand(fixed_batch - len(x), -1, -1, -1)])
        with torch.no_grad():
            y = model(x.to(DEVICE))[0][: len(images)].float().cpu().numpy()
        return [postprocess(y[i], *prepared[i][1:]) for i in range(len(images))]

    return predict

In [ ]:
float_det = load_float(FLOAT_WEIGHTS)
results = {
    "float (torch)": evaluate(
        torch_predictor(YoloQAT(copy.deepcopy(float_det), IMGSZ).eval().to(DEVICE))
    )
}
print(results)

# PTQ baseline: the same quantizer, calibrated on training images without training. Shows what QAT buys.
ptq = make_qat_graph(float_det, BATCH)
move_exported_model_to_eval(ptq)
ptq = ptq.to(DEVICE)
calibrate = torch_predictor(ptq, BATCH)
for start in range(0, min(len(TRAIN_IMAGES), 256), BATCH):  # observers record activation ranges
    calibrate([cv2.imread(str(p)) for p in TRAIN_IMAGES[start : start + BATCH]])
ptq.apply(disable_observer)
results["int8 PTQ (fake-quant)"] = evaluate(torch_predictor(ptq, BATCH))
print(results)
del ptq

In [ ]:
cfg = get_cfg(overrides={"imgsz": IMGSZ, "mosaic": 0.0, "mixup": 0.0, "batch": BATCH, "workers": 4})
data = check_det_dataset(str(DATA_YAML))
train_set = build_yolo_dataset(cfg, data["train"], BATCH, data, mode="train", stride=32)
loader = build_dataloader(train_set, BATCH, cfg.workers, shuffle=True)

loss_model = copy.deepcopy(float_det).to(DEVICE)
loss_model.args = cfg  # box / cls / dfl loss gains
loss_fn = v8DetectionLoss(loss_model)

qat = make_qat_graph(float_det, BATCH).to(DEVICE)
optimizer = torch.optim.AdamW(
    [p for p in qat.parameters() if p.requires_grad], lr=QAT_LR, weight_decay=5e-4
)
steps = max(QAT_EPOCHS * (len(loader) - 1), 1)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=QAT_LR, total_steps=steps, pct_start=0.1
)
best = -1.0

for epoch in range(QAT_EPOCHS):
    freeze = epoch >= QAT_EPOCHS - QAT_FREEZE_EPOCHS
    if freeze:
        qat.apply(disable_observer)
        move_exported_model_to_eval(qat)  # BatchNorm uses running stats; fake-quant stays on
    else:
        qat.apply(enable_observer)
        move_exported_model_to_train(qat)
    total, count, started = 0.0, 0, time.monotonic()
    for batch in loader:
        if batch["img"].shape[0] != BATCH:  # the graph is traced at a fixed batch size
            continue
        loss, items = loss_fn(qat(batch["img"].to(DEVICE).float() / 255), batch)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(qat.parameters(), 10.0)
        optimizer.step()
        if scheduler.last_epoch < steps:
            scheduler.step()
        total, count = total + float(items.sum()), count + 1
    move_exported_model_to_eval(qat)
    qat.apply(disable_observer)  # validation images must not move the scales
    metrics = evaluate(torch_predictor(qat, BATCH))
    print(
        f"epoch {epoch + 1}/{QAT_EPOCHS}  loss {total / max(count, 1):.3f}  mAP50 {metrics['mAP50']:.3f}  "
        f"mAP50-95 {metrics['mAP50-95']:.3f}  {'frozen ' if freeze else ''}{time.monotonic() - started:.0f}s"
    )
    if freeze and metrics["mAP50-95"] > best:  # keep the best epoch once the scales are fixed
        best, results["int8 QAT (fake-quant)"] = metrics["mAP50-95"], metrics
        torch.save({k: v.cpu() for k, v in qat.state_dict().items()}, QAT_DIR / "qat_state.pt")

print(results)

## 3. Export the int8 TFLite for the QCS6490

The QAT state goes into a batch-1 copy of the graph, which is converted to real int8 ops and handed to
ai-edge-torch with the same quantizer. The model takes **NHWC** input `[1, IMGSZ, IMGSZ, 3]` (RGB, float
0..1, quantized by the first op) and returns only the decoded `[1, 4 + nc, N]` — the Ultralytics layout
that `aether_vision.detector` decodes, with normalized boxes. NMS stays in the detector.

Why it suits the Hexagon HTP (v68) on the QCS6490: static shapes and batch 1; int8 weights (symmetric,
per-channel) and int8 activations throughout, including the box decode, so the QNN delegate can take the
whole graph instead of bouncing to the CPU; only ops the HTP supports (conv, SiLU, sigmoid, softmax,
concat, resize, pooling).

In [ ]:
TFLITE_PATH = OUT / "model_int8.tflite"
LABELS_PATH = OUT / "labels.txt"


class DecodedOnly(torch.nn.Module):
    def __init__(self, graph: torch.nn.Module) -> None:
        super().__init__()
        self.graph = graph

    def forward(self, x):
        return self.graph(x)[0]


deploy = make_qat_graph(load_float(FLOAT_WEIGHTS), batch=1)
deploy.load_state_dict(torch.load(QAT_DIR / "qat_state.pt"), strict=True)
move_exported_model_to_eval(deploy)
deploy.apply(disable_observer)
int8_graph = convert_pt2e(deploy, fold_quantize=False)

nhwc = ai_edge_torch.to_channel_last_io(DecodedOnly(int8_graph), args=[0])
edge_model = ai_edge_torch.convert(
    nhwc, (torch.rand(1, IMGSZ, IMGSZ, 3),), quant_config=QuantConfig(pt2e_quantizer=quantizer)
)
edge_model.export(str(TFLITE_PATH))
LABELS_PATH.write_text("\n".join(NAMES) + "\n")
print(f"{TFLITE_PATH}  ({TFLITE_PATH.stat().st_size / 1e6:.1f} MB)")

## 4. Verify

The TFLite runs in LiteRT on the CPU here, with the same pre- and post-processing as
`aether_vision.detector`. It should land within about a point of the QAT fake-quant result; a large gap
means the conversion changed something.

In [ ]:
from ai_edge_litert.interpreter import Interpreter

interpreter = Interpreter(model_path=str(TFLITE_PATH), num_threads=4)
interpreter.allocate_tensors()
input_detail = interpreter.get_input_details()[0]
output_details = interpreter.get_output_details()
for d in [input_detail, *output_details]:
    print(d["name"], d["shape"], d["dtype"].__name__, d["quantization"])
dtypes = [t["dtype"] for t in interpreter.get_tensor_details()]
print(
    f"tensors: int8 {sum(t == np.int8 for t in dtypes)}, float32 {sum(t == np.float32 for t in dtypes)}, "
    f"int32 {sum(t == np.int32 for t in dtypes)}"
)
assert len(output_details) == 1
assert output_details[0]["shape"][:2].tolist() == [1, 4 + len(NAMES)]
assert input_detail["shape"].tolist() == [1, IMGSZ, IMGSZ, 3]


def tflite_predict(images):
    """Same input quantization and output decoding as aether_vision.detector."""
    out = []
    for image in images:
        rgb, scale, pad_x, pad_y = letterbox(image)
        dtype, (in_scale, zero_point) = input_detail["dtype"], input_detail["quantization"]
        if dtype == np.float32:
            data = rgb.astype(np.float32) / 255
        else:
            info = np.iinfo(dtype)
            data = np.clip(np.round(rgb / 255 / in_scale + zero_point), info.min, info.max).astype(
                dtype
            )
        interpreter.set_tensor(input_detail["index"], data[None])
        interpreter.invoke()
        detail = output_details[0]
        raw = interpreter.get_tensor(detail["index"])[0].astype(np.float32)
        o_scale, o_zero = detail["quantization"]
        out.append(
            postprocess((raw - o_zero) * o_scale if o_scale > 0 else raw, scale, pad_x, pad_y)
        )
    return out


results["int8 TFLite (LiteRT)"] = evaluate(tflite_predict)
for name, m in results.items():
    print(f"{name:24s} mAP50 {m['mAP50']:.3f}   mAP50-95 {m['mAP50-95']:.3f}")
(OUT / "results.json").write_text(json.dumps(results, indent=2))

samples = random.sample(VAL_IMAGES, min(8, len(VAL_IMAGES)))
frames = [cv2.imread(str(p)) for p in samples]
show_grid([draw(f, *p) for f, p in zip(frames, tflite_predict(frames))])

### Optional: profile on a real QCS6490 in Qualcomm AI Hub

[AI Hub](https://aihub.qualcomm.com) runs the model on a QCS6490 and reports latency and which layers ran
on the NPU. Needs a free account: add your API token as a Kaggle secret named `QAI_HUB_API_TOKEN`
(*Add-ons → Secrets*). Every layer should be on `NPU`; layers on `CPU` are ops the HTP rejected.

In [ ]:
RUN_AI_HUB = False

if RUN_AI_HUB:
    import subprocess
    import sys

    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "qai-hub"], check=True)

    import qai_hub as hub

    if ON_KAGGLE:
        from kaggle_secrets import UserSecretsClient

        token = UserSecretsClient().get_secret("QAI_HUB_API_TOKEN")
        subprocess.run(
            ["qai-hub", "configure", "--api_token", token], check=True, capture_output=True
        )
    job = hub.submit_profile_job(
        model=str(TFLITE_PATH),
        device=hub.Device("QCS6490 (Proxy)"),
        options="--compute_unit npu",
        name="aether-vision",
    )
    profile = job.download_profile()
    summary = profile["execution_summary"]
    print(
        f"inference {summary['estimated_inference_time'] / 1000:.2f} ms, "
        f"peak memory {summary['estimated_inference_peak_memory'] / 1e6:.1f} MB"
    )
    units: dict[str, int] = {}
    for layer in profile["execution_detail"]:
        units[layer["compute_unit"]] = units.get(layer["compute_unit"], 0) + 1
    print("layers per compute unit:", units)

## 5. Download

`aether_model.zip` holds the model, the labels, the metrics and the float weights. On the robot:

```bash
unzip aether_model.zip -d models/
```

then set `detector.model_path` to `models/model_int8.tflite` and `detector.labels_path` to
`models/labels.txt` in the dashboard. The float `best.pt` is a good annotator for the next capture
session: `annotate.py label --model best.pt`.

In [ ]:
archive_path = WORK / "aether_model.zip"
with zipfile.ZipFile(archive_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in (TFLITE_PATH, LABELS_PATH, OUT / "results.json", FLOAT_WEIGHTS):
        archive.write(path, path.name)
print(f"{archive_path} ({archive_path.stat().st_size / 1e6:.1f} MB)")